In [0]:
from pyspark.sql import functions as F
bronze_df = spark.read.table("clinical_trials.bronze_studies")

# Bronze Delta Table을 읽는다.
# Bronze에는 ClinicalTrials.gov의 원본 study 구조를 최대한 보존해 두었고,
# Silver에서는 분석과 재사용이 쉬운 형태로 정제·정규화한다.

print(bronze_df.count())
# bronze_df.printSchema()

In [0]:
# ClinicalTrials.gov의 nested JSON에서 임상시험 단위의 핵심 필드를 추출한다.
# 하나의 study가 하나의 row가 되도록 구성해,
# downstream 분석 및 다른 데이터셋과의 join이 쉬운 Silver 테이블을 만든다.

silver_studies_df = bronze_df.select(
    F.col("study.protocolSection.identificationModule.nctId").alias("nct_id"),
    F.col("study.protocolSection.identificationModule.briefTitle").alias("brief_title"),
    F.col("study.protocolSection.identificationModule.officialTitle").alias(
        "official_title"
    ),
    F.col("study.protocolSection.designModule.studyType").alias("study_type"),
    F.col("study.protocolSection.designModule.phases").alias("phases"),
    F.col("study.protocolSection.statusModule.overallStatus").alias("overall_status"),
    F.col("study.protocolSection.designModule.enrollmentInfo.count")
    .cast("long")
    .alias("enrollment_count"),
    # 원본 데이터의 값을 분석 목적에 맞는 데이터 타입으로 변환한다.
    # 숫자와 날짜 타입을 명확히 지정해 이후 집계·필터링·날짜 연산을 안정적으로 수행할 수 있게 한다.
    F.col("study.protocolSection.designModule.enrollmentInfo.type").alias(
        "enrollment_type"
    ),
    F.col("study.protocolSection.designModule.designInfo.allocation").alias(
        "allocation"
    ),
    F.col("study.protocolSection.designModule.designInfo.interventionModel").alias(
        "intervention_model"
    ),
    F.col("study.protocolSection.designModule.designInfo.primaryPurpose").alias(
        "primary_purpose"
    ),
    F.col("study.protocolSection.designModule.designInfo.maskingInfo.masking").alias(
        "masking"
    ),
    F.to_date(F.col("study.protocolSection.statusModule.startDateStruct.date")).alias(
        "start_date"
    ),
    F.to_date(
        F.col("study.protocolSection.statusModule.completionDateStruct.date")
    ).alias("completion_date"),
    F.to_date(
        F.col("study.protocolSection.statusModule.lastUpdatePostDateStruct.date")
    ).alias("last_update_date"),
    F.col("study.hasResults").alias("has_results"),
    F.col("source"),
    F.col("ingested_at"),
    F.col("study.protocolSection.descriptionModule.briefSummary").alias(
        "brief_summary"
    ),
    F.col("study.protocolSection.designModule.designInfo.maskingInfo.whoMasked").alias(
        "who_masked"
    ),
    F.col("study.protocolSection.eligibilityModule.eligibilityCriteria").alias(
        "eligibility_criteria"
    ),
)

display(silver_studies_df.limit(20))

# Silver 변환 결과에 기본 데이터 품질 규칙을 적용한다.
# study 단위 테이블이므로 Bronze와 row 수가 동일해야 하며,
# NCT ID는 null이거나 중복되어서는 안 된다.

print("Bronze:", bronze_df.count())
print("Silver:", silver_studies_df.count())

# NCT ID null 확인
silver_studies_df.filter(F.col("nct_id").isNull()).count()

# NCT ID 중복 확인
duplicates_df = silver_studies_df.groupBy("nct_id").count().filter(F.col("count") > 1)

display(duplicates_df)

In [0]:
(
    silver_studies_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("clinical_trials.silver_studies")
)

spark.sql("""
SELECT COUNT(*)
FROM clinical_trials.silver_studies
""").show()

In [0]:
# 한 study에 여러 condition이 존재할 수 있으므로 array를 explode하여
# Study-Condition의 1:N 관계를 별도 Silver 테이블로 정규화한다.
# 이를 통해 질환별 임상시험 수 등의 분석과 join이 쉬워진다.

silver_conditions_df = (
    bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id"),

    # explode_outer : 배열이 없거나 null인 경우도 안전하게 처리합니다.
        F.explode_outer(
            "study.protocolSection.conditionsModule.conditions"
        ).alias("condition")
    )
    .filter(F.col("condition").isNotNull())

    # 실제 condition이 없는 row는 제거
)

display(silver_conditions_df.limit(30))

print("Study rows:", silver_studies_df.count())
print("Condition rows:", silver_conditions_df.count())

In [0]:
# 같은 임상시험에 같은 condition이 두 번 들어가 있다면 탐지
condition_duplicates = (
    silver_conditions_df
    .groupBy("nct_id", "condition")
    .count()
    .filter(F.col("count") > 1)
)

display(condition_duplicates)


# 중복 확인 시, 왜 중복이 생겼는지 확인 후 필요하다면 중복 제거
# silver_conditions_df = silver_conditions_df.dropDuplicates(
#     ["nct_id", "condition"]
# )

In [0]:
(
    silver_conditions_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clinical_trials.silver_conditions")
)